# Fase 1 — Extração de Características
**Dataset:** CBIS-DDSM (massa, treino)  
**Features:** ResNet50 (ImageNet) + Pyradiomics  
**Saída:** `features_treino.csv`

In [4]:
# ── Bibliotecas ──────────────────────────────────────────────────────────────
import os
import glob
import warnings

import numpy as np
import pandas as pd
import cv2
import SimpleITK as sitk

from radiomics import featureextractor
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

warnings.filterwarnings('ignore')

# ── Configuração global ───────────────────────────────────────────────────────
# True  → processa apenas as 5 primeiras linhas (modo de teste rápido)
# False → processa o dataset completo
MODO_TESTE = True

CSV_TREINO    = 'csv/mass_case_description_train_set.csv'
PASTA_JPEG    = 'jpeg'
ARQUIVO_SAIDA = 'features_treino.csv'

print('Bibliotecas carregadas com sucesso.')

ModuleNotFoundError: No module named 'radiomics'

In [ ]:
# ── Conversão de caminho CSV → arquivo .jpg local ─────────────────────────────
#
# O CSV registra caminhos DICOM com 3 níveis:
#   PastaPaciente / UID1 / UID2 / 000000.dcm
#
# Na pasta jpeg/ os arquivos estão organizados como:
#   jpeg / UID2 / {N}-{dim}.jpg
#
# onde N = int('000000') + 1  →  000000.dcm → 1-*.jpg, 000001.dcm → 2-*.jpg

def converter_caminho(caminho_csv: str, base_dir: str = PASTA_JPEG):
    """
    Converte um caminho .dcm do CSV para o .jpg correspondente em base_dir.
    Retorna None se o arquivo não for encontrado.
    """
    partes = caminho_csv.replace('\\', '/').split('/')

    if len(partes) < 3:
        return None

    uid_pasta  = partes[-2]                              # ex: 1.3.6.1.4.1…
    nome_dcm   = partes[-1]                              # ex: 000000.dcm
    indice_jpg = int(nome_dcm.replace('.dcm', '')) + 1   # 000000 → 1

    padrao  = os.path.join(base_dir, uid_pasta, f'{indice_jpg}-*.jpg')
    matches = glob.glob(padrao)

    return matches[0] if matches else None


# Smoke-test com a primeira linha do CSV
_df_test = pd.read_csv(CSV_TREINO, nrows=1)
_caminho  = converter_caminho(_df_test['cropped image file path'].iloc[0])
print('Teste de conversão de caminho:')
print(f'  Encontrado: {_caminho is not None}  →  {_caminho}')

In [ ]:
# ── Pré-processamento de imagem (OpenCV) ──────────────────────────────────────

def preprocessar_imagem(caminho: str):
    """
    Lê a imagem em escala de cinza e aplica CLAHE.
    Retorna None se o arquivo não existir ou falhar ao abrir.
    """
    img = cv2.imread(caminho, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(img)


def ler_mascara(caminho: str):
    """
    Lê a máscara ROI e a binariza (0 ou 255).
    Retorna None se o arquivo não existir ou falhar ao abrir.
    """
    mask = cv2.imread(caminho, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    _, mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    return mask_bin


print('Funções de pré-processamento definidas.')

In [ ]:
# ── ResNet50 (features de representação profunda) ─────────────────────────────
#
# include_top=False + pooling='avg' → vetor de 2048 valores por imagem

modelo_resnet = ResNet50(weights='imagenet', include_top=False, pooling='avg')
print(f'ResNet50 carregada. Dimensão de saída: {modelo_resnet.output_shape}')


def extrair_features_resnet(img_cinza: np.ndarray) -> np.ndarray:
    """
    Converte a imagem CLAHE (escala de cinza) para RGB 224×224
    e extrai features usando o modelo ResNet50.
    Retorna um vetor 1-D com 2048 floats.
    """
    img_rgb   = cv2.cvtColor(img_cinza, cv2.COLOR_GRAY2RGB)
    img_224   = cv2.resize(img_rgb, (224, 224))
    img_batch = np.expand_dims(img_224.astype(np.float32), axis=0)
    img_prep  = preprocess_input(img_batch)
    features  = modelo_resnet.predict(img_prep, verbose=0)
    return features.flatten()

In [ ]:
# ── Pyradiomics (features radiômicas) ────────────────────────────────────────

_params = {
    'setting': {
        'force2D': True,
        'force2Ddimension': 0,
    },
    'featureClass': {
        'firstorder': [],
        'glcm':       [],
        'shape2D':    [],
    },
    'imageType': {'Original': {}},
}

_extrator = featureextractor.RadiomicsFeatureExtractor(_params)
print('Extrator Pyradiomics inicializado.')


def extrair_features_radiomics(img_cinza: np.ndarray, mask_bin: np.ndarray) -> dict:
    """
    Extrai features radiômicas (firstorder, glcm, shape2D) na região da máscara.
    Retorna apenas os valores numéricos (descarta metadados de diagnóstico).
    """
    sitk_img  = sitk.GetImageFromArray(img_cinza.astype(np.float32))
    sitk_mask = sitk.GetImageFromArray((mask_bin > 0).astype(np.uint8))

    resultado = _extrator.execute(sitk_img, sitk_mask)

    return {
        k: float(v)
        for k, v in resultado.items()
        if not k.startswith('diagnostics_')
        and isinstance(v, (int, float, np.floating))
    }

In [ ]:
# ── Loop principal de processamento e exportação ──────────────────────────────

df = pd.read_csv(CSV_TREINO)

if MODO_TESTE:
    df = df.head(5)
    print(f'MODO_TESTE ativo — processando apenas {len(df)} imagens.\n')
else:
    print(f'Processando dataset completo: {len(df)} imagens.\n')

resultados = []
total = len(df)

for num_atual, (_, linha) in enumerate(df.iterrows(), start=1):
    print(f'Processando imagem {num_atual}/{total}...')

    # ── Resolução dos caminhos ──────────────────────────────────────────────
    caminho_crop = converter_caminho(linha['cropped image file path'])
    caminho_mask = converter_caminho(linha['ROI mask file path'])

    if caminho_crop is None:
        print(f'  [AVISO] Crop não encontrado para imagem {num_atual}. Pulando...')
        continue
    if caminho_mask is None:
        print(f'  [AVISO] Máscara não encontrada para imagem {num_atual}. Pulando...')
        continue

    # ── Pré-processamento ───────────────────────────────────────────────────
    img_clahe = preprocessar_imagem(caminho_crop)
    mask_bin  = ler_mascara(caminho_mask)

    if img_clahe is None:
        print(f'  [AVISO] Falha ao abrir imagem: {caminho_crop}. Pulando...')
        continue
    if mask_bin is None:
        print(f'  [AVISO] Falha ao abrir máscara: {caminho_mask}. Pulando...')
        continue

    # ── Extração de features ────────────────────────────────────────────────
    feats_resnet = extrair_features_resnet(img_clahe)
    dict_resnet  = {f'resnet_{i}': v for i, v in enumerate(feats_resnet)}

    dict_rad = extrair_features_radiomics(img_clahe, mask_bin)

    # ── Montagem do registro ────────────────────────────────────────────────
    registro = {**dict_resnet, **dict_rad, 'pathology': linha['pathology']}
    resultados.append(registro)

    print(f'  OK — {len(dict_resnet)} features ResNet50  +  {len(dict_rad)} features Pyradiomics')

# ── Exportação ──────────────────────────────────────────────────────────────
df_features = pd.DataFrame(resultados)
df_features.to_csv(ARQUIVO_SAIDA, index=False)

print(f'\nConcluído! {len(df_features)} imagens processadas com sucesso.')
print(f'Arquivo salvo em: {ARQUIVO_SAIDA}')
print(f'Shape do DataFrame: {df_features.shape}')